# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imranusmaneii/flyrank-ml-imran/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys

# Navigate to repo root (this notebook lives in work/notebooks/)
while not os.path.isdir('data/raw') and os.getcwd() != '/':
    os.chdir('..')

assert os.path.exists('data/raw/content_refresh_anonymized.csv'), 'starter CSV not found'
print('Working dir:', os.getcwd())

Working dir: D:\FlyRank AI\Week 1\Task #5\flyrank-ml-imran


## 1. My lane (or freestyle) and why

**Lane 4 — CTR / Engagement Opportunity Scoring**

This lane asks: *Which visible pages under-capture clicks or engagement and deserve metadata, content, or monitoring review?*

Three things in the starter data make this lane the strongest fit:

1. **CTR varies 6x by position tier** — pages at `page_1` average CTR of 0.35 while `deep` pages average 0.055. This gap is the single clearest signal in the dataset and creates a natural position-adjusted baseline: any useful CTR opportunity score must beat a simple tier-average comparison.

2. **Half of striking-distance pages underperform on CTR** — of 6,436 pages at `top_3` or `striking` positions, 3,208 (49.8%) sit below the median CTR for their position tier. These are pages with enough visibility to matter but weak click-through — exactly the kind of page an editor can act on by rewriting titles, meta descriptions, or snippet structure.

3. **The existing starter pipeline already covers decline prediction** (Lane 2 territory) with decent results. Lane 4 is a genuinely different angle — it focuses on *click efficiency* rather than *traffic trajectory*, producing a different kind of ranked action list that an editor can use alongside, not instead of, the decline queue.

Lane 4 also scales well: the warehouse's 79M daily rows add time-window CTR trends, position drift, and query-mix context that the starter snapshot cannot provide. The 7-week arc has a clear progression from starter-dataset EDA to warehouse-grade modeling.

In [2]:
import pandas as pd
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print(f'Dataset: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Unique clients: {df["client_id"].nunique()}')
print(f'Columns relevant to CTR analysis: ctr, avg_position, position_tier, impressions_90d, engagement_rate, scroll_rate')

Dataset: 30,000 rows x 44 columns
Unique clients: 32
Columns relevant to CTR analysis: ctr, avg_position, position_tier, impressions_90d, engagement_rate, scroll_rate


## 2. The question: decision, action, cost of a wrong call

**Decision:** Which visible pages should an SEO editor review first for CTR and engagement improvement — specifically, pages that under-capture clicks relative to their position tier?

**Action:** An editor receives a ranked list of pages with reason codes (e.g., "high impressions, strong position, weak CTR") and reviews the top candidates for title/meta description rewrites, snippet structure changes, or intent-matching improvements.

**Cost of a wrong call:**
- *False positive* (page flagged but already performing normally for its position): the editor wastes 15-30 minutes reviewing a page that needs no change. At scale, this erodes trust in the system and the editor stops checking the queue.
- *False negative* (high-opportunity page missed): the page continues to underperform, leaving clicks on the table. If the page has 1,000 impressions and a CTR of 0.05 when peers at the same position average 0.25, that is roughly 200 missed clicks per measurement period.

The cost asymmetry favors precision: wasting editor time is worse than missing one opportunity, because the editor's capacity is finite and trust is fragile.

In [3]:
# Quantify the decision scope
visible = df[df['impressions_90d'] >= 100]
striking = visible[visible['position_tier'].isin(['top_3', 'striking'])]
median_ctr = striking['ctr'].median()
underperformers = striking[striking['ctr'] < median_ctr]

print(f'Pages with enough visibility to review (impressions >= 100): {len(visible):,}')
print(f'Pages at striking-distance positions (top_3 + striking): {len(striking):,}')
print(f'Of those, below-median CTR ({median_ctr:.4f}): {len(underperformers):,} ({len(underperformers)/len(striking)*100:.1f}%)')

Pages with enough visibility to review (impressions >= 100): 22,006
Pages at striking-distance positions (top_3 + striking): 6,436
Of those, below-median CTR (0.1600): 3,208 (49.8%)


## 3. Quick look at the data (2-3 real numbers)

Three numbers from the starter dataset that justify this lane for the next 7 weeks:

In [4]:
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
visible = df[df['impressions_90d'] >= 100].copy()

# --- Number 1: CTR gap by position tier ---
ctr_by_pos = visible.groupby('position_tier')['ctr'].mean().sort_values(ascending=False)
top_tier = ctr_by_pos.iloc[0]
bottom_tier = ctr_by_pos.iloc[-1]
gap_ratio = top_tier / bottom_tier

print('Number 1 — CTR varies dramatically by position tier')
print(ctr_by_pos.round(4).to_string())
print(f'\nPage_1 ({top_tier:.4f}) vs deep ({bottom_tier:.4f}) = {gap_ratio:.1f}x gap')
print('=> Position-adjusted analysis is essential; a single CTR threshold would be meaningless.')
print()

# --- Number 2: Half of striking-distance pages underperform ---
striking = visible[visible['position_tier'].isin(['top_3', 'striking'])]
median_ctr = striking['ctr'].median()
underperformers = striking[striking['ctr'] < median_ctr]

print('Number 2 — Large addressable opportunity in striking-distance positions')
print(f'{len(striking):,} pages at top_3 or striking positions')
print(f'{len(underperformers):,} ({len(underperformers)/len(striking)*100:.1f}%) have below-median CTR')
print(f'Median CTR at these positions: {median_ctr:.4f}')
print('=> Roughly half of visible pages in good positions are CTR-underperformers.')
print()

# --- Number 3: Enough volume across content types for feature diversity ---
ct_counts = visible['content_type'].value_counts()
print('Number 3 — Content type diversity across visible pages')
print(ct_counts.to_string())
print(f'\n=> Multiple content types to condition on, preventing a one-size-fits-all CTR rule.')

Number 1 — CTR varies dramatically by position tier
position_tier
page_1      0.3548
top_3       0.3341
striking    0.2558
page_3_5    0.1424
deep        0.0554

Page_1 (0.3548) vs deep (0.0554) = 6.4x gap
=> Position-adjusted analysis is essential; a single CTR threshold would be meaningless.

Number 2 — Large addressable opportunity in striking-distance positions
6,436 pages at top_3 or striking positions
3,208 (49.8%) have below-median CTR
Median CTR at these positions: 0.1600
=> Roughly half of visible pages in good positions are CTR-underperformers.

Number 3 — Content type diversity across visible pages
content_type
keyword article       21288
comparison article      366
feedly article          352

=> Multiple content types to condition on, preventing a one-size-fits-all CTR rule.


## 4. Careful words: what I can and can't claim

**What this work can support:**

- *Observational:* We can observe that CTR varies by position tier and that a large fraction of visible pages underperform their tier's expected CTR. These are measured facts in the data.
- *Directional:* We can suggest that pages with high impressions, strong position, and low CTR are candidates for title/meta review. This is a reasonable hypothesis supported by the signal pattern.
- *Decision-support:* The ranked CTR opportunity list is a tool to help editors prioritize their limited review time. It does not make the decision for them.

**What this work cannot claim:**

- We are *not* proving that low CTR causes poor ranking, or that improving CTR will improve position. The data is cross-sectional (one snapshot); we cannot establish causation.
- We are *not* predicting Google's algorithm or AI search behavior. CTR is one observed signal among many.
- We are *not* claiming that every low-CTR page needs fixing. Some pages naturally have low CTR for their topic, intent, or SERP features (e.g., featured snippets that satisfy without a click).
- We are *not* claiming that a refresh will recover traffic. That would require a controlled experiment.

**A real limitation — not just a hedge:**

Precision@50, the metric I named in the frame, requires ground-truth labels that *editor-confirmed* a page was or was not worth reviewing. Those labels do not exist yet. Right now the best I can do is define a proxy label (e.g., CTR below tier median) and measure how well the model ranks pages according to that proxy. The proxy is reasonable but it is not the same as an editor saying "yes, this page needed work." Until a human reviews a sample of the top-ranked pages and confirms or rejects the recommendations, Precision@50 is a *ranking hypothesis*, not a validated metric. This is the single biggest gap between where the project is now and where it needs to be by capstone.

In [5]:
# Sanity check: the proxy label we can measure today vs the real label we need
visible = df[df['impressions_90d'] >= 100].copy()
striking = visible[visible['position_tier'].isin(['top_3', 'striking'])]
median_ctr = striking['ctr'].median()

proxy_label_count = len(striking[striking['ctr'] < median_ctr])
total_striking = len(striking)

print(f'Proxy label (CTR < tier median): {proxy_label_count} pages out of {total_striking}')
print(f'This proxy is measurable today. Editor-confirmed labels are not.')
print(f'Gap to close: recruit editor review of top-50 ranked pages to get real Precision@50.')

Proxy label (CTR < tier median): 3208 pages out of 6436
This proxy is measurable today. Editor-confirmed labels are not.
Gap to close: recruit editor review of top-50 ranked pages to get real Precision@50.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.